# FLUJO 3 — Resultados PSBP-FD
1. Lee trazas generadas por MATLAB
2. Diagnósticos de convergencia MCMC
3. Evaluación predictiva

## 1. Imports y rutas

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os
import sys
import inspect
from pathlib import Path
from datetime import datetime
from scipy.io import loadmat

# Módulos del paquete
from model_psbp_fd.pipelines import (
    FunctionalDomain,
    FARSimulator,
    gaussian_integral_kernel,
    build_integral_matrix,
)
from model_psbp_fd.functions_models import DataStandardizer
from model_psbp_fd.functions_models import FunctionalRepresentation
from model_psbp_fd.utils import get_project_root
from model_psbp_fd.models.psbp_fd_v2.functions.predict import PSBPPredictor

# Módulos de visualización
from model_psbp_fd.graphics import (
    plot_traces_bj, plot_traces_pj,
    plot_convergence_bj, plot_convergence_pj,
    plot_global_components, plot_active_clusters,
    plot_empirical_sample, plot_functional_mean,
    plot_functional_variance, plot_mean_and_variance,
    plot_fts_empirical, plot_fts_functional, plot_fts_comparison,
    plot_scatter_theta, plot_functional_comparison,
    plot_diagnostico_estandarizacion, plot_seleccion_basis,
    plot_fpca_scree, plot_fpca_correlacion_lag0, plot_rezagos_heatmap,
)

plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline


In [5]:
# ── Raíz del proyecto ─────────────────────────────────────────────────────
PROJECT_ROOT = get_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT : {PROJECT_ROOT}")

# ── Identificación del experimento ───────────────────────────────────────
BASENAME      = "modelo_unificado"
TT            = 1
SEED          = 4123
EXPERIMENT_ID = f"{BASENAME}_{TT}"
print(f"Experiment ID : {EXPERIMENT_ID}")
print(f"Seed (base)   : {SEED}")


PROJECT_ROOT : C:\Users\56jua\OneDrive\Desktop\git_tesis\bayesian_non_parametrics-1
Experiment ID : modelo_unificado_1
Seed (base)   : 4123


In [6]:
# ── Construcción de rutas ─────────────────────────────────────────────────
_REPORT_DIR   = PROJECT_ROOT / "reports"   / "simulaciones" / EXPERIMENT_ID
_ARTEFACT_DIR = PROJECT_ROOT / "artefact"  / "simulaciones" / EXPERIMENT_ID

PATHS = {
    "raw":          PROJECT_ROOT / "data" / "simulaciones" / "raw"       / EXPERIMENT_ID,
    "functional":   PROJECT_ROOT / "data" / "simulaciones" / "processed" / "functional" / EXPERIMENT_ID,
    "predict":      PROJECT_ROOT / "data" / "simulaciones" / "processed" / "predict"    / EXPERIMENT_ID,
    "out_report":   _REPORT_DIR,
    "out_artefact": _ARTEFACT_DIR,
}
for name, path in PATHS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"  {name:12s} → {path}")


  raw          → C:\Users\56jua\OneDrive\Desktop\git_tesis\bayesian_non_parametrics-1\data\simulaciones\raw\modelo_unificado_1
  functional   → C:\Users\56jua\OneDrive\Desktop\git_tesis\bayesian_non_parametrics-1\data\simulaciones\processed\functional\modelo_unificado_1
  predict      → C:\Users\56jua\OneDrive\Desktop\git_tesis\bayesian_non_parametrics-1\data\simulaciones\processed\predict\modelo_unificado_1
  out_report   → C:\Users\56jua\OneDrive\Desktop\git_tesis\bayesian_non_parametrics-1\reports\simulaciones\modelo_unificado_1
  out_artefact → C:\Users\56jua\OneDrive\Desktop\git_tesis\bayesian_non_parametrics-1\artefact\simulaciones\modelo_unificado_1


## 2. Lectura de datasets funcionales y manifest FPCA
Cargamos los datasets AR(p) generados por el pipeline de preprocesamiento,
el manifest con `component_idx` y `n_lags`, y la representación funcional ajustada.

In [7]:
# ── Manifest ─────────────────────────────────────────────────────────────
manifest_path = PATHS["functional"] / "datasets_manifest.json"
assert manifest_path.exists(), f"No se encontró: {manifest_path}"
with open(manifest_path) as f:
    manifest = json.load(f)

n_lags        = manifest["n_lags"]
COMPONENT_IDX = manifest["component_idx"]      # lista base-0
n_components  = len(COMPONENT_IDX)
cov_names     = manifest["cov_names"]

print(f"n_components  : {n_components}")
print(f"n_lags        : {n_lags}")
print(f"COMPONENT_IDX : {COMPONENT_IDX}")
print(f"cov_names     : {cov_names}")


n_components  : 9
n_lags        : 2
COMPONENT_IDX : [0, 1, 2, 3, 4, 5, 6, 7, 8]
cov_names     : ['fpc_1_lag1', 'fpc_2_lag1', 'fpc_3_lag1', 'fpc_4_lag1', 'fpc_5_lag1', 'fpc_6_lag1', 'fpc_7_lag1', 'fpc_8_lag1', 'fpc_9_lag1', 'fpc_1_lag2', 'fpc_2_lag2', 'fpc_3_lag2', 'fpc_4_lag2', 'fpc_5_lag2', 'fpc_6_lag2', 'fpc_7_lag2', 'fpc_8_lag2', 'fpc_9_lag2']


In [8]:
# ── Cargar datasets AR(p) en dfs[k] ─────────────────────────────────────
# dfs[k] es el DataFrame del componente k (respuesta + predictores rezagados)
dfs = {}
for k in range(n_components):
    fpc_idx = COMPONENT_IDX[k] + 1          # base-1
    fpath   = PATHS["functional"] / f"dataset_fpc_{fpc_idx}.csv"
    assert fpath.exists(), f"No se encontró: {fpath}"
    dfs[k]  = pd.read_csv(fpath)
    print(f"  FPC {fpc_idx}  shape={dfs[k].shape}  cols={list(dfs[k].columns)}")


  FPC 1  shape=(118, 19)  cols=['fpc_1', 'fpc_1_lag1', 'fpc_2_lag1', 'fpc_3_lag1', 'fpc_4_lag1', 'fpc_5_lag1', 'fpc_6_lag1', 'fpc_7_lag1', 'fpc_8_lag1', 'fpc_9_lag1', 'fpc_1_lag2', 'fpc_2_lag2', 'fpc_3_lag2', 'fpc_4_lag2', 'fpc_5_lag2', 'fpc_6_lag2', 'fpc_7_lag2', 'fpc_8_lag2', 'fpc_9_lag2']
  FPC 2  shape=(118, 19)  cols=['fpc_2', 'fpc_1_lag1', 'fpc_2_lag1', 'fpc_3_lag1', 'fpc_4_lag1', 'fpc_5_lag1', 'fpc_6_lag1', 'fpc_7_lag1', 'fpc_8_lag1', 'fpc_9_lag1', 'fpc_1_lag2', 'fpc_2_lag2', 'fpc_3_lag2', 'fpc_4_lag2', 'fpc_5_lag2', 'fpc_6_lag2', 'fpc_7_lag2', 'fpc_8_lag2', 'fpc_9_lag2']
  FPC 3  shape=(118, 19)  cols=['fpc_3', 'fpc_1_lag1', 'fpc_2_lag1', 'fpc_3_lag1', 'fpc_4_lag1', 'fpc_5_lag1', 'fpc_6_lag1', 'fpc_7_lag1', 'fpc_8_lag1', 'fpc_9_lag1', 'fpc_1_lag2', 'fpc_2_lag2', 'fpc_3_lag2', 'fpc_4_lag2', 'fpc_5_lag2', 'fpc_6_lag2', 'fpc_7_lag2', 'fpc_8_lag2', 'fpc_9_lag2']
  FPC 4  shape=(118, 19)  cols=['fpc_4', 'fpc_1_lag1', 'fpc_2_lag1', 'fpc_3_lag1', 'fpc_4_lag1', 'fpc_5_lag1', 'fpc_6_lag

In [9]:
# ── Cargar representación funcional ajustada ─────────────────────────────
# FunctionalRepresentation debe estar serializada en PATHS["functional"]
fr_path = PATHS["functional"] / "functional_representation.npz"
if fr_path.exists():
    fr = FunctionalRepresentation.load(fr_path)
    print(f"✓ FunctionalRepresentation cargada: método={fr.method}, K={fr.n_basis_fpca}")
else:
    print(f"⚠ No se encontró functional_representation.npz en {PATHS['functional']}")
    print("  Las secciones de reconstrucción funcional no estarán disponibles.")
    fr = None

# ── Cargar grilla del dominio ────────────────────────────────────────────
grid_path = PATHS["functional"] / "domain_grid.npy"
if grid_path.exists():
    domain_grid = np.load(grid_path)
    print(f"✓ Grilla cargada: {domain_grid.shape}")
else:
    print(f"⚠ No se encontró domain_grid.npy; usando grilla por defecto [0,1] con 100 pts")
    domain_grid = np.linspace(0, 1, 100)

# ── Cargar datos estandarizados completos ────────────────────────────────
X_std_path = PATHS["functional"] / "X_std.npy"
if X_std_path.exists():
    X_std = np.load(X_std_path)
    print(f"✓ X_std cargado: {X_std.shape}  (T, G)")
else:
    print(f"⚠ No se encontró X_std.npy")
    X_std = None


⚠ No se encontró functional_representation.npz en C:\Users\56jua\OneDrive\Desktop\git_tesis\bayesian_non_parametrics-1\data\simulaciones\processed\functional\modelo_unificado_1
  Las secciones de reconstrucción funcional no estarán disponibles.
⚠ No se encontró domain_grid.npy; usando grilla por defecto [0,1] con 100 pts
⚠ No se encontró X_std.npy


## 3. Lectura de trazas MATLAB
Configuramos el patrón de nombre, definimos el loader y ensamblamos
`models_chains[k][c]` con el adaptador `MatlabTraceModel`.

In [11]:
# ── Configuración de trazas ───────────────────────────────────────────────
TRACE_DIR = PATHS["out_artefact"]
N_ITER    = 1    # número de iteraciones del experimento (réplicas por componente)
                 # Ajusta este valor según cuántas cadenas haya generado MATLAB

# Patrón de nombre: chain_fpc_<fpc_idx>_iter<tt>.mat
def trace_path(fpc_idx: int, tt: int) -> Path:
    return TRACE_DIR / f"chain_fpc_{fpc_idx}_iter{tt:02d}.mat"

print(f"TRACE_DIR : {TRACE_DIR}")
print(f"N_ITER    : {N_ITER}  (cadenas por componente)")
print(f"Ejemplo   : {trace_path(1, 1).name}")


TRACE_DIR : C:\Users\56jua\OneDrive\Desktop\git_tesis\bayesian_non_parametrics-1\artefact\simulaciones\modelo_unificado_1
N_ITER    : 1  (cadenas por componente)
Ejemplo   : chain_fpc_1_iter01.mat


In [ ]:
# ── Loader: .mat → dict de trazas ────────────────────────────────────────
def load_trace_mat(path: Path) -> tuple:
    """
    Carga un archivo .mat generado por psbp_train y devuelve
    (traces_dict, burn, feature_names_list).
    """
    m      = loadmat(str(path))
    keys2d = ["betajhout", "beta0hout", "tauhout", "alphahout",
              "psijhout", "Gammajhout", "gammajhout", "pijout",
              "wjout", "osumout", "inEout"]
    traces = {k: np.asarray(m[k], dtype=np.float64) for k in keys2d}
    for k in ["muout", "N1out", "Nout"]:
        traces[k] = np.asarray(m[k], dtype=np.float64).ravel()
    burn  = int(np.asarray(m["burn"]).ravel()[0])
    feat  = str(np.atleast_1d(m["feature_names"]).ravel()[0]).split(",")
    return traces, burn, feat


# ── Adaptador: trazas MATLAB → objeto compatible con funciones de visualización
class MatlabTraceModel:
    """
    Expone .traces y .feature_names_ tal como los esperan las funciones
    plot_* del paquete, y delega predicción en PSBPPredictor.
    """
    def __init__(self, traces: dict, burn: int, feature_names: list):
        self.traces         = traces
        self.feature_names_ = list(feature_names)
        self.burn           = int(burn)
        self.n_features_    = int(traces["betajhout"].shape[2])
        # PSBPPredictor es agnóstico a la escala (no desestandariza)
        self.predictor_     = PSBPPredictor(traces=traces, burn=burn)

    def _design(self, df: pd.DataFrame) -> np.ndarray:
        """Construye matriz de diseño con intercepto desde un DataFrame."""
        Xp = np.asarray(df.iloc[:, 1:], dtype=float)
        return np.hstack([np.ones((Xp.shape[0], 1)), Xp])

    def predict(self, df: pd.DataFrame, return_std: bool = False):
        return self.predictor_.predict(self._design(df), return_std=return_std)

    def inclusion_probs(self, as_series: bool = False):
        incl = self.predictor_.inclusion_probs()
        if as_series:
            return pd.Series(incl, index=self.feature_names_, name="inclusion_prob")
        return incl

    def rmse(self, df: pd.DataFrame, y_obs=None) -> float:
        if y_obs is None:
            y_obs = np.asarray(df.iloc[:, 0], dtype=float)
        return self.predictor_.rmse(self._design(df), y_obs)


In [ ]:
# ── Ensamblar models_chains[k][c] ────────────────────────────────────────
models_chains = {k: {} for k in range(n_components)}
BURN = None

for k in range(n_components):
    fpc_idx = COMPONENT_IDX[k] + 1
    for c in range(N_ITER):
        tt   = c + 1
        path = trace_path(fpc_idx, tt)
        if not path.exists():
            print(f"  ⚠ No encontrado: {path.name}  — se omite")
            continue

        traces, burn, feat = load_trace_mat(path)

        # Verificar consistencia con las columnas del dataset
        expected = list(dfs[k].columns[1:])
        if feat != expected:
            print(f"  [aviso k={k} tt={tt}] feature_names del .mat ≠ columnas del dataset")
            print(f"    mat={feat}")
            print(f"    dfs={expected}")

        models_chains[k][c] = MatlabTraceModel(traces, burn, feat)
        BURN = burn
        nsim = traces["betajhout"].shape[0]
        N    = traces["betajhout"].shape[1]
        print(f"  FPC {fpc_idx}  iter{tt:02d}  ← {path.name}  "
              f"(p={len(feat)}, N={N}, nsim={nsim}, burn={burn})")

# Filtrar componentes con al menos una cadena cargada
models_chains = {k: v for k, v in models_chains.items() if len(v) > 0}
n_chains_loaded = len(next(iter(models_chains.values())))

TRACE_META = {
    "nsim":     int(nsim),
    "burn":     int(BURN),
    "N":        int(N),
    "n_chains": n_chains_loaded,
    "source":   "matlab",
}
print(f"\n✓ models_chains listo: {len(models_chains)} componente(s) × {n_chains_loaded} cadena(s)  BURN={BURN}")


## 4. Diagnósticos de convergencia MCMC

| Estadístico | Referencia | Lectura |
|---|---|---|
| ACF(lag) | autocorrelación residual | cerca de 0 ⇒ buena mezcla |
| ESS | Geyer (1992) | ESS ≳ 400 ideal |
| Geweke z | test z entre segmentos | \|z\| < 2 ⇒ no rechazo |
| Gelman-Rubin R̂ | Brooks-Gelman (1998) | R̂ < 1.1 ⇒ convergencia |


### 4.1 Componentes globales (μ, N₁, α, τ, β₀, p_j)

In [ ]:
for k in models_chains:
    fig = plot_global_components(
        models_chains, k, BURN, n_chains_loaded,
        title_prefix = f"FPC {k+1}",
        save_path    = str(PATHS["out_report"] / f"10_global_comp_k{k+1}.png"),
    )
    plt.show()


In [ ]:
for k in models_chains:
    fig = plot_active_clusters(
        models_chains, k, BURN, n_chains_loaded,
        title_prefix = f"FPC {k+1}",
        save_path    = str(PATHS["out_report"] / f"11_clusters_activos_k{k+1}.png"),
    )
    plt.show()


### 4.2 Trazas β_j y p_j por variable

In [ ]:
for k in models_chains:
    feat = models_chains[k][0].feature_names_
    fig  = plot_traces_bj(
        models_chains, k, BURN, n_chains_loaded,
        feature_names = feat,
        title_prefix  = f"FPC {k+1}",
        save_path     = str(PATHS["out_report"] / f"12_trazas_bj_k{k+1}.png"),
    )
    plt.show()


In [ ]:
for k in models_chains:
    feat = models_chains[k][0].feature_names_
    fig  = plot_traces_pj(
        models_chains, k, BURN, n_chains_loaded,
        feature_names = feat,
        title_prefix  = f"FPC {k+1}",
        save_path     = str(PATHS["out_report"] / f"13_trazas_pj_k{k+1}.png"),
    )
    plt.show()


### 4.3 Convergencia completa (traza | ACF | posterior + métricas)

In [ ]:
all_diag_bj = {}
for k in models_chains:
    feat = models_chains[k][0].feature_names_
    fig, diag = plot_convergence_bj(
        models_chains, k, BURN, n_chains_loaded,
        feature_names = feat,
        title_prefix  = f"FPC {k+1}",
        save_path     = str(PATHS["out_report"] / f"14_convergencia_bj_k{k+1}.png"),
    )
    all_diag_bj[k] = diag
    plt.show()


In [ ]:
all_diag_pj = {}
for k in models_chains:
    feat = models_chains[k][0].feature_names_
    fig, diag = plot_convergence_pj(
        models_chains, k, BURN, n_chains_loaded,
        feature_names = feat,
        title_prefix  = f"FPC {k+1}",
        save_path     = str(PATHS["out_report"] / f"15_convergencia_pj_k{k+1}.png"),
    )
    all_diag_pj[k] = diag
    plt.show()


In [ ]:
# ── Tabla resumen de diagnósticos ────────────────────────────────────────
diag_records = []
for k in models_chains:
    for rec in all_diag_bj.get(k, []):
        diag_records.append({"componente": k+1, "param": "beta_j", **rec})
    for rec in all_diag_pj.get(k, []):
        diag_records.append({"componente": k+1, "param": "p_j",    **rec})

diag_df = pd.DataFrame(diag_records)
if not diag_df.empty:
    display(
        diag_df.style
        .format({"ess_min": "{:.1f}", "ess_mean": "{:.1f}",
                 "geweke_max": "{:+.2f}", "rhat": "{:.3f}"})
        .background_gradient(subset=["rhat"],    cmap="RdYlGn_r", vmin=1.0, vmax=1.2)
        .background_gradient(subset=["ess_min"], cmap="RdYlGn",   vmin=0,   vmax=500)
        .map(lambda v: "font-weight:bold;color:#c0392b" if v is False else "",
             subset=["converge"])
        .set_caption("Diagnósticos MCMC por variable")
    )
else:
    print("No hay datos de diagnóstico disponibles.")


## 5. Evaluación predictiva

Usando la cadena 0 (primera iteración) de cada componente como modelo principal.
Para análisis multi-cadena, promediar o seleccionar por R̂.

In [ ]:
# ── Predicciones por componente ───────────────────────────────────────────
# Usamos la cadena c=0 de cada componente (iter01)
eval_results = {}

for k in models_chains:
    model = models_chains[k][0]
    df_k  = dfs[k]

    y_obs = np.asarray(df_k.iloc[:, 0], dtype=float)
    y_hat = model.predict(df_k)

    incl_named = model.inclusion_probs(as_series=True)
    rmse_k     = model.rmse(df_k, y_obs)

    eval_results[k] = {
        "y_obs":       y_obs,
        "y_hat":       y_hat,
        "incl_named":  incl_named,
        "rmse":        rmse_k,
    }
    print(f"  FPC {COMPONENT_IDX[k]+1}  RMSE={rmse_k:.6f}")

print("\n✓ eval_results construido")


### 5.1 Scatter θ observado vs θ predicho (scores FPCA)

In [ ]:
fig = plot_scatter_theta(
    eval_results,
    n_components = len(models_chains),
    save_path    = str(PATHS["out_report"] / "20_scatter_theta.png"),
)
plt.show()


### 5.2 Probabilidades de inclusión por variable

In [ ]:
# Tabla de inclusión por componente y variable
incl_df = pd.DataFrame({
    f"FPC {COMPONENT_IDX[k]+1}": eval_results[k]["incl_named"]
    for k in models_chains
})
display(
    incl_df.style
    .background_gradient(cmap="RdYlGn", vmin=0, vmax=1)
    .format("{:.3f}")
    .set_caption("P(γ_j = 1 | data) — Probabilidades de inclusión por variable y componente")
)


### 5.3 Reconstrucción funcional (si fr disponible)

In [ ]:
if fr is not None and X_std is not None:
    fig, metrics = plot_functional_comparison(
        eval_results  = eval_results,
        fr            = fr,
        domain_grid   = domain_grid,
        X_true        = X_std,
        n_components  = len(models_chains),
        n_lags        = n_lags,
        n_snapshots   = 5,
        save_path     = str(PATHS["out_report"] / "21_reconstruccion_funcional.png"),
    )
    plt.show()

    print("\n── Métricas de reconstrucción ──────────────────────────────────")
    for key, val in metrics.items():
        print(f"  {key:20s} : {val:.6f}")
else:
    print("⚠ fr o X_std no disponibles — omitiendo reconstrucción funcional.")


### 5.4 Serie de tiempo funcional predicha vs verdadera (si fr disponible)

In [ ]:
if fr is not None and X_std is not None:
    # ── Score matrices ────────────────────────────────────────────────────
    THETA_obs  = np.column_stack([eval_results[k]["y_obs"] for k in models_chains])
    THETA_pred = np.column_stack([eval_results[k]["y_hat"] for k in models_chains])

    X_repr_obs  = fr.reconstruct(THETA_obs)    # (T_eff, G)
    X_repr_pred = fr.reconstruct(THETA_pred)   # (T_eff, G)
    X_true_eff  = X_std[n_lags:, :]            # (T_eff, G)

    # ── Curvas empíricas vs reconstrucción PSBP ──────────────────────────
    fig = plot_fts_comparison(
        X          = X_true_eff,
        grid       = domain_grid,
        fr         = fr,
        save_path  = str(PATHS["out_report"] / "22_fts_comparacion.png"),
    )
    plt.show()
else:
    print("⚠ fr o X_std no disponibles — omitiendo plot de series funcionales.")


### 5.5 Métricas resumen

In [ ]:
# ── Tabla de métricas por componente ─────────────────────────────────────
rows = []
for k in models_chains:
    r = eval_results[k]
    ss_res = np.sum((r["y_obs"] - r["y_hat"]) ** 2)
    ss_tot = np.sum((r["y_obs"] - r["y_obs"].mean()) ** 2)
    r2     = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    corr   = np.corrcoef(r["y_obs"], r["y_hat"])[0, 1]
    rows.append({
        "FPC":  f"FPC {COMPONENT_IDX[k]+1}",
        "RMSE": r["rmse"],
        "R²":   r2,
        "Corr": corr,
        "T_eff": len(r["y_obs"]),
    })

metrics_df = pd.DataFrame(rows).set_index("FPC")
display(
    metrics_df.style
    .format({"RMSE": "{:.6f}", "R²": "{:.4f}", "Corr": "{:.4f}"})
    .background_gradient(subset=["RMSE"], cmap="RdYlGn_r")
    .background_gradient(subset=["R²"],   cmap="RdYlGn",   vmin=0, vmax=1)
    .set_caption("Métricas predictivas por componente FPCA")
)
